In [1]:
#importing libraries

import os
from dotenv import load_dotenv
import json
from openai import OpenAI

load_dotenv()

client=OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Multi Tool Agent environment is ready.")

Multi Tool Agent environment is ready.


In [2]:
#Creating 4 mathematical tools

def add(a,b):
    return a+b

def subtract(a,b):
    return a-b

def multiply(a,b):
    return a*b

def divide(a,b):
    if b==0:
        raise ValueError("Cannot divide by 0.")
    return a/b

In [3]:
print(add(10, 5))
print(subtract(10, 5))
print(multiply(10, 5))
print(divide(10, 5))

15
5
50
2.0


In [4]:
#Registering the tools with the model == Instead of defining one tool schema at a time, we can register all four.

def make_tool(name, description):
    return {
        "type": "function",
        "name": name,
        "description": description,
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "First number"
                },
                "b": {
                    "type": "number",
                    "description": "Second number"
                }
            },
            "required": ["a", "b"],
            "additionalProperties": False
        },
        "strict": True
    }

In [5]:
#Creating tool list
tools = [
    make_tool("add", "Add two numbers."),
    make_tool("subtract", "Subtract the second number from the first."),
    make_tool("multiply", "Multiply two numbers."),
    make_tool("divide", "Divide the first number by the second.")
]

In [6]:
#Inspecting the tool
for tool in tools:
    print(tool["name"], "-", tool["description"])

add - Add two numbers.
subtract - Subtract the second number from the first.
multiply - Multiply two numbers.
divide - Divide the first number by the second.


In [7]:
#Creating a tool registry. We need a safety way to connect the requested function name to our Python function
tool_registry = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
    "divide": divide
}

In [8]:
function_name = "multiply"

function = tool_registry[function_name]

result = function(10, 5)

print(result)

50


In [9]:
#Building the tool execution function

def execute_tool(tool_name, arguments):

    if tool_name not in tool_registry:
        raise ValueError(f"Unknown tool: {tool_name}")

    if set(arguments.keys()) != {"a", "b"}:
        raise ValueError("Invalid arguments.")

    a = arguments["a"]
    b = arguments["b"]

    if isinstance(a, bool) or not isinstance(a, (int, float)):
        raise ValueError("Argument a must be a number.")

    if isinstance(b, bool) or not isinstance(b, (int, float)):
        raise ValueError("Argument b must be a number.")

    function = tool_registry[tool_name]

    return function(a, b)

In [10]:
#testing
print(execute_tool("add", {"a": 20, "b": 10}))
print(execute_tool("multiply", {"a": 6, "b": 7}))

30
42


In [11]:
execute_tool("divide", {"a": 10, "b": 0})

ValueError: Cannot divide by 0.

In [12]:
#Let the model choose a tool
response = client.responses.create(
    model="gpt-4.1-mini",
    instructions="""
    You are a calculator assistant.
    Use the available tools for mathematical operations.
    """,
    input="What is 45 multiplied by 12?",
    tools=tools,
    tool_choice="auto"
)

In [13]:
for item in response.output:
    print("Type:", item.type)

    if item.type == "function_call":
        print("Tool:", item.name)
        print("Arguments:", item.arguments)
        print("Call ID:", item.call_id)

Type: function_call
Tool: multiply
Arguments: {"a":45,"b":12}
Call ID: call_ffTJhFTLioTsT9p9Lcfm8NY5


In [14]:
#Multi tool executiion loop
def run_agent(question):

    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions="""
        You are a helpful calculator agent.

        Use the available tools for calculations.
        You may request multiple tools when necessary.
        Explain the final result clearly.
        """,
        input=question,
        tools=tools,
        tool_choice="auto"
    )

    while True:

        function_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]

        if not function_calls:
            return response.output_text

        tool_outputs = []

        for item in function_calls:

            try:
                arguments = json.loads(item.arguments)

                result = execute_tool(
                    item.name,
                    arguments
                )

                output = {
                    "result": result,
                    "error": None
                }

            except Exception as error:

                output = {
                    "result": None,
                    "error": str(error)
                }

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps(output)
            })

        response = client.responses.create(
            model="gpt-4.1-mini",
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools
        )

In [15]:
print(run_agent("What is 25 multiplied by 16?"))

25 multiplied by 16 is 400.


In [16]:
print(
    run_agent(
        "Add 25 and 15, then multiply the result by 3."
    )
)

The result of adding 25 and 15 is 40. Multiplying this result by 3 gives 120.


In [ ]:
print(
    run_agent(
        "Subtract 20 from 100, then divide the result by 4."
    )
)
